In [1]:
# Copyright 2023 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Generate descriptions from videos

## Overview

This notebook shows how to generate descriptions of videos in a GCS bucket.  
It uses the [Youtube UGC dataset](https://media.withyoutube.com/) and uses the [Gemini](https://cloud.google.com/vertex-ai/docs/generative-ai/model-reference/gemini) to generate video descriptions for each video.

#### **Steps**
Using Spark,
1) It reads the table [Youtube UGC dataset](https://media.withyoutube.com/) from gs://dataproc-metastore-public-binaries/youtube_ucg/
2) It calls Vertex AI Gemini API vision pro to generate description from videos.

### Setup

Make sure the service account running this notebook has the required permissions:

- **Run the notebook**
  - AI Platform Notebooks Service Agent
  - Notebooks Admin
  - Vertex AI Administrator
- **Read files from bucket**
  - Storage Object Viewer
- **Run Dataproc jobs**
  - Dataproc Service Agent
  - Dataproc Worker
- **Call Google APIs**
  - Service Usage Consumer

#### Imports

In [2]:
import os
import sys
import time

# Ensure Spark driver and workers use the same Python interpreter (avoids PYTHON_VERSION_MISMATCH)
os.environ.setdefault("PYSPARK_PYTHON", sys.executable)
os.environ.setdefault("PYSPARK_DRIVER_PYTHON", sys.executable)

from pyspark.sql.functions import regexp_replace, concat
from pyspark.sql.functions import udf, col, lit

import google.auth
import google.auth.transport.requests
import requests

import pandas as pd
pd.set_option('display.max_colwidth', None)
pd.set_option('display.min_rows', 20)

In [3]:
# When using Dataproc Serverless, installed packages are automatically available on all nodes
%pip install --upgrade google-cloud-aiplatform -q
# When using a Dataproc cluster, you will need to install these packages during cluster creation: https://cloud.google.com/dataproc/docs/tutorials/python-configuration

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 7.36.2 which is incompatible.



[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: /Users/garvey/Dev/EventRelay/.venv/bin/python -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


#### Authentication

In [4]:
credentials, project_id = google.auth.default()
auth_req = google.auth.transport.requests.Request()
credentials.refresh(auth_req)
PROJECT_ID = project_id
LOCATION = "global"

#### Setup Spark Session

In [5]:
from pyspark.sql import SparkSession

In [6]:
import os

# Hadoop/Spark requires Java 17 or 21 (Java 24+ removed Subject.getSubject)
for jdk_path in ("/opt/homebrew/opt/openjdk@17", "/opt/homebrew/opt/openjdk@21"):
    if os.path.exists(jdk_path) and "JAVA_HOME" not in os.environ:
        os.environ["JAVA_HOME"] = jdk_path
        os.environ["PATH"] = f"{jdk_path}/bin:{os.environ.get('PATH', '')}"
        break

spark = SparkSession.builder \
  .appName("Video descriptions generation") \
  .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/16 05:23:20 WARN Utils: Your hostname, Garvey.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.90 instead (on interface en0)
26/11/16 05:23:20 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/11/16 05:23:20 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


#### Read dataset

In [7]:
BINARIES_BUCKET_PATH = "gs://dataproc-metastore-public-binaries/youtube_ucg/"

try:
    binaries_df = spark.read.format("binaryFile").option("recursiveFileLookup", "true").load(BINARIES_BUCKET_PATH)
except Exception as e:
    # Fallback for local Spark without Hadoop GCS connector
    from google.cloud import storage
    bucket_name, prefix = BINARIES_BUCKET_PATH.replace("gs://", "", 1).split("/", 1)
    client = storage.Client()
    gcs_paths = [
        (f"gs://{bucket_name}/{blob.name}",)
        for blob in client.list_blobs(bucket_name, prefix=prefix, max_results=20)
        if not blob.name.endswith("/")
    ]
    binaries_df = spark.createDataFrame(gcs_paths, ["path"])

26/11/16 05:23:22 WARN FileSystem: Cannot load filesystem
java.util.ServiceConfigurationError: org.apache.hadoop.fs.FileSystem: Provider org.apache.hadoop.fs.viewfs.ViewFileSystem could not be instantiated
	at java.base/java.util.ServiceLoader.fail(ServiceLoader.java:552)
	at java.base/java.util.ServiceLoader$ProviderImpl.newInstance(ServiceLoader.java:712)
	at java.base/java.util.ServiceLoader$ProviderImpl.get(ServiceLoader.java:672)
	at java.base/java.util.ServiceLoader$2.next(ServiceLoader.java:1256)
	at org.apache.hadoop.fs.FileSystem.loadFileSystems(FileSystem.java:3525)
	at org.apache.hadoop.fs.FileSystem.getFileSystemClass(FileSystem.java:3562)
	at org.apache.hadoop.fs.FsUrlStreamHandlerFactory.<init>(FsUrlStreamHandlerFactory.java:77)
	at org.apache.spark.sql.internal.SharedState$.liftedTree2$1(SharedState.scala:209)
	at org.apache.spark.sql.internal.SharedState$.org$apache$spark$sql$internal$SharedState$$setFsUrlStreamHandlerFactory(SharedState.scala:208)
	at org.apache.spark.

26/11/16 05:23:22 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: gs://dataproc-metastore-public-binaries/youtube_ucg/.
java.lang.UnsupportedOperationException: getSubject is not supported
	at java.base/javax.security.auth.Subject.getSubject(Subject.java:277)
	at org.apache.hadoop.security.UserGroupInformation.getCurrentUser(UserGroupInformation.java:588)
	at org.apache.hadoop.fs.FileSystem$Cache$Key.<init>(FileSystem.java:3888)
	at org.apache.hadoop.fs.FileSystem$Cache$Key.<init>(FileSystem.java:3878)
	at org.apache.hadoop.fs.FileSystem$Cache.get(FileSystem.java:3666)
	at org.apache.hadoop.fs.FileSystem.get(FileSystem.java:557)
	at org.apache.hadoop.fs.Path.getFileSystem(Path.java:366)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:57)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:384)
	at org.apache.spark.sql.catalyst.analysis.Re

In [8]:
# Let's select the paths of the first 5 youtube videos
paths_df = binaries_df.select("path").limit(5)

#### Define UDF and call Gemini API to generate video descriptions

In [9]:
response_schema = {
    "type": "object",
    "properties": {
        "where": {"type": "string"},
        "how_many_people": {"type": "integer"},
        "task": {"type": "string"},
        "proposition": {"type": "string"},
        "description": {"type": "string"}
    },
    "required": ["where","how_many_people","task","proposition","description"],
}

system_instruction = "\n".join([
    """Format the 5 items as attributes of a JSON object: where, how_many_people, task, proposition and description.""",
    """The response should be a single valid formatted JSON object only."""
])

prompt = f"""
    Create a short description for this video with the following questions:
     1) Where is the video recorded?
     2) How many people are shown in the video?
     3) What the people are doing in the video?
     4) Whats the proposition for the video, i.e what it is about?
     5) A sumary description from the itens 1,2,3 and 4
    """

In [10]:
from google import genai
from google.genai import types

def predict(uri, prompt, system_instruction=system_instruction, response_schema=response_schema, content_type="video/mp4", temperature=1, model_name="gemini-3-flash-preview", project_id=PROJECT_ID, location=LOCATION):
    client = genai.Client(vertexai=True, project=project_id, location=location)

    response = client.models.generate_content(
        model=model_name,
        contents=[
            types.Part.from_uri(file_uri=uri, mime_type=content_type),
            prompt,
        ],
        config=types.GenerateContentConfig(
            temperature=temperature,
            system_instruction=system_instruction,
            response_mime_type="application/json",
            response_schema=response_schema,
        ),
    )

    return response.text

In [11]:
def generate_descriptions(gcs_uri):

    descriptions = predict(gcs_uri, prompt)
    return descriptions

generate_descriptions_udf = udf(generate_descriptions)

In [12]:
df_descriptions = paths_df.sort(paths_df.path.asc()).withColumn("data", generate_descriptions_udf(paths_df.path))

In [13]:
df_descriptions.cache()

DataFrame[path: string, data: string]

In [14]:
df_descriptions.toPandas()

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


,path,data
0,gs://dataproc-metastore-public-binaries/youtube_ucg/original_videos/HowTo/360P/HowTo_360P-041c.mp4,"{""where"": ""a woodworking workshop"", ""how_many_people"": 3, ""task"": ""participating in a video conference"", ""proposition"": ""a live online discussion among woodworkers or makers"", ""description"": ""The video shows three individuals engaged in a Google Hangout session. The primary speaker is recorded from a workshop filled with various tools, suggesting the group is holding an online discussion about woodworking or DIY projects.""}"
1,gs://dataproc-metastore-public-binaries/youtube_ucg/original_videos/HowTo/360P/HowTo_360P-0562.mp4,"{\n ""where"": ""A television studio for a shopping channel called Jewellery Maker."",\n ""how_many_people"": 1,\n ""task"": ""The presenter is demonstrating and describing a set of chocolate agate slices to the audience."",\n ""proposition"": ""A sales pitch for 700ct Chocolate Agate Slices (5pcs/set) priced at £7.95."",\n ""description"": ""In a Jewellery Maker television studio, a female presenter is showcasing a set of dyed chocolate agate slices. She holds a sample close to the camera to highlight its texture and color, providing details about the item LZPO70 and its price of £7.95 to encourage viewers to purchase the product.""\n}"
2,gs://dataproc-metastore-public-binaries/youtube_ucg/original_videos/HowTo/360P/HowTo_360P-06be.mp4,"{\n ""where"": ""indoor"",\n ""how_many_people"": 2,\n ""task"": ""drawing a portrait"",\n ""proposition"": ""tutorial on how to draw a portrait of a young girl"",\n ""description"": ""In an indoor setting, a hand is shown sketching a realistic pencil portrait of a young girl. The video includes a small inset displaying the reference photo of the girl being drawn.""\n}"
3,gs://dataproc-metastore-public-binaries/youtube_ucg/original_videos/HowTo/360P/HowTo_360P-09f8.mp4,"{\n ""where"": ""A screen recording of an Android mobile phone interface."",\n ""how_many_people"": 0,\n ""task"": ""The user is navigating through various mobile phone apps and checking video folders before displaying a call to action."",\n ""proposition"": ""A short screen recording clip, likely a test or a template for a video, concluding with a request for viewers to leave a comment."",\n ""description"": ""The video is a screen capture of an Android smartphone with no people visible. It shows a user navigating from the home screen into a file manager, briefly looking at video folders, and returning to the main screen. The clip concludes with a colorful particle effect and the word 'comment' displayed on a blue screen.""\n}"
4,gs://dataproc-metastore-public-binaries/youtube_ucg/original_videos/HowTo/360P/HowTo_360P-127f.mp4,"{""where"":""a television studio for the JewelleryMaker shopping channel"",""how_many_people"":1,""task"":""decorating a Christmas tree with jewelry and presenting items for sale"",""proposition"":""promoting and selling rainbow-coated pyrite jewelry items"",""description"":""In a television studio setting, a single male presenter demonstrates rainbow-coated pyrite beads by decorating a Christmas tree with them and presenting the jewelry to the camera for a sales broadcast.""}"


#### Extract feature from generated text

In [15]:
from pyspark.sql.functions import from_json, col, regexp_replace
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
schema = StructType(
    [
        StructField('where', StringType(), True),
        StructField('how_many_people', IntegerType(), True),
        StructField('proposition', StringType(), True),
        StructField('description', StringType(), True),
        StructField('task', StringType(), True)
    ]
)
clean_json = regexp_replace(col("data"), r"^```(?:json)?\s*|\s*```$", "")
df_final = df_descriptions.withColumn("exploded_data", from_json(clean_json, schema))\
    .select(col('path'), col('exploded_data.*'))

In [16]:
df_final.toPandas()

,path,where,how_many_people,proposition,description,task
0,gs://dataproc-metastore-public-binaries/youtube_ucg/original_videos/HowTo/360P/HowTo_360P-041c.mp4,a woodworking workshop,3,a live online discussion among woodworkers or makers,"The video shows three individuals engaged in a Google Hangout session. The primary speaker is recorded from a workshop filled with various tools, suggesting the group is holding an online discussion about woodworking or DIY projects.",participating in a video conference
1,gs://dataproc-metastore-public-binaries/youtube_ucg/original_videos/HowTo/360P/HowTo_360P-0562.mp4,A television studio for a shopping channel called Jewellery Maker.,1,A sales pitch for 700ct Chocolate Agate Slices (5pcs/set) priced at £7.95.,"In a Jewellery Maker television studio, a female presenter is showcasing a set of dyed chocolate agate slices. She holds a sample close to the camera to highlight its texture and color, providing details about the item LZPO70 and its price of £7.95 to encourage viewers to purchase the product.",The presenter is demonstrating and describing a set of chocolate agate slices to the audience.
2,gs://dataproc-metastore-public-binaries/youtube_ucg/original_videos/HowTo/360P/HowTo_360P-06be.mp4,indoor,2,tutorial on how to draw a portrait of a young girl,"In an indoor setting, a hand is shown sketching a realistic pencil portrait of a young girl. The video includes a small inset displaying the reference photo of the girl being drawn.",drawing a portrait
3,gs://dataproc-metastore-public-binaries/youtube_ucg/original_videos/HowTo/360P/HowTo_360P-09f8.mp4,A screen recording of an Android mobile phone interface.,0,"A short screen recording clip, likely a test or a template for a video, concluding with a request for viewers to leave a comment.","The video is a screen capture of an Android smartphone with no people visible. It shows a user navigating from the home screen into a file manager, briefly looking at video folders, and returning to the main screen. The clip concludes with a colorful particle effect and the word 'comment' displayed on a blue screen.",The user is navigating through various mobile phone apps and checking video folders before displaying a call to action.
4,gs://dataproc-metastore-public-binaries/youtube_ucg/original_videos/HowTo/360P/HowTo_360P-127f.mp4,a television studio for the JewelleryMaker shopping channel,1,promoting and selling rainbow-coated pyrite jewelry items,"In a television studio setting, a single male presenter demonstrates rainbow-coated pyrite beads by decorating a Christmas tree with them and presenting the jewelry to the camera for a sales broadcast.",decorating a Christmas tree with jewelry and presenting items for sale
